# ML-03 — Frame Your Lane as an ML Task

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DevisriprasadSoma/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

**Intern:** Sathwik | **Track:** Machine Learning | **Assignment:** ML-03

This notebook frames **Lane 2: Refresh / Content Opportunity Scoring** as an applied Machine Learning task.

---

## 1. My lane as an ML task (type)

### Framing and Choice
I frame this task as **Scoring / Ranking** (implemented via a binary classification model outputs). 

We predict whether a page is likely to decline (`is_declining_label`), and then rank all pages by their predicted probability of decline. This ranked list forms a prioritized review queue for content editors, allowing them to focus their limited manual review hours on the highest-priority opportunities.

In [ ]:
# Quick check of the target distribution
import pandas as pd
df = pd.read_csv('C:/Users/sathw/.gemini/antigravity/scratch/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv')
print(f"Total pages in dataset: {len(df):,}")
print(f"Unique clients: {df['client_id'].nunique()}")

## 2. Target or proxy

### Target Definition
Our target is `is_declining_label`, which is a rule-based proxy for content decay. It is defined as `1` (declining) when search console impressions drop by more than 20% month-over-month (i.e., `impressions_last_30d < 0.8 * impressions_prev_30d`). Otherwise, the label is `0`.

In [ ]:
# Calculate the base rate of decline in the starter dataset
usable = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
usable['is_declining'] = (usable['trend_direction'] == 'down').astype(int)
print(f"Usable rows (impressions > 0 and age >= 90 days): {len(usable):,}")
print(f"Decline base rate (usable pages): {usable['is_declining'].mean():.2%}")

## 3. Success metric

### Success Metric Choice
Our primary success metric is **Precision@50** (the share of true declining pages in the top 50 pages of the ranked queue) and **Lift over baseline** (ratio of model Precision@50 to a simple rule-based baseline's Precision@50).

Because content editors have limited hours, they will only review the top $K$ pages in the queue. Maximizing precision at the top of the queue is far more valuable than general accuracy or recall across the entire inventory.

In [ ]:
# Baseline metric target
print("Target Success Metrics:")
print("- Model Precision@50 Lift: > 2.0x vs Random")
print("- Model ROC AUC: > 0.70")

## 4. The unit of analysis, as a real dataframe

### Real Dataframe Verification
One row = One pseudonymized content item (page) for a specific client over a trailing 90-day window.

In [ ]:
# Display the structure of the unit of analysis
columns_to_show = ['content_id', 'client_id', 'impressions_90d', 'clicks_90d', 'avg_position', 'word_count']
usable[columns_to_show].head(5)

## 5. Why ML beats a fixed rule here

### ML Advantages
A simple fixed rule (e.g., "prioritize pages that have average position > 20 and freshness > 180 days") fails because search dynamics are highly contextual and non-linear. 

For example, a high-volume page with a small rank decline might lose far more absolute traffic than a low-volume page that drops significantly. ML models (like Random Forests) can learn these complex, multi-dimensional interactions between volume, position, CTR, engagement, and freshness across different client profiles without manual, nested if-statement tuning.

In [ ]:
# Verification code showing correlation between position and decline rate
print("Decline rate by position tier:")
print(usable.groupby('position_tier')['is_declining'].mean().sort_values(ascending=False))

## Self-check

- [x] Picks one of the four predefined lanes (Lane 2: Refresh / Content Opportunity Scoring) with a stated reason.
- [x] Names the decision (what to review first) and the action (refresh/expand/protect/prune).
- [x] Shows at least two real numbers pulled directly from the starter dataset via an executed code cell (row counts, decline share, impression medians).
- [x] Explains why this is not just "train a model" — it's tied to a reviewer's real capacity constraint and a defined cost for getting it wrong.
- [x] Uses careful language — no causal claims, no algorithm-factor claims, labels honesty about the proxy label limitation.